# AMPS I · Week 10 — Lab
## The heat equation: θ-method, stability, non-smooth data, Cahn–Hilliard

| | |
|---|---|
| **Lecture sections** | Part B (implementation, Gaussian benchmark, temporal rates) · Part C (explicit limit, non-smooth data) · Part D (Cahn–Hilliard, LLM) |
| **Time** | ≈ 80 min |
| **You will** | write a θ-method loop that assembles and factorises once · measure first- and second-order temporal convergence and see the spatial floor · find the explicit stability limit from the eigenvalues and watch it hide · see Crank–Nicolson oscillate and fix it · run Cahn–Hilliard and check energy and mass |

$u_t=\Delta u$ on $\Omega=(-2,2)^2$. Benchmark: $u=\frac{1}{1+4at}\exp\bigl(-\frac{a(x^2+y^2)}{1+4at}\bigr)$, $a=5$, with its own values as Dirichlet data, $T=0.1$.

**Reminder.** `%%fenicsx` cells run in a separate MPI process; ordinary cells have NumPy and matplotlib only. Data passes through `/tmp/fnx/`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · The θ-method loop  *(lecture §B1–B3)*
Read `heat` in the cell: the matrix $M+\theta\Delta tK$ is assembled and factorised **once** per $(\Delta t,\theta)$; each step updates $t$ and the boundary
function, assembles the right-hand side, solves, and **copies** $u_h$ into $u_n$. Backward Euler, P2, $N=64$, 20 steps; profile on $y=0$.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

AG, T = 5.0, 0.1

def heat(N, k, dts, thetas, init="gauss"):
    """theta-method, one step per entry of dts/thetas. init='gauss': exact Gaussian with its own Dirichlet data;
    'box': indicator of (-1/2,1/2)^2 with u = 0. Returns (L2 error at the end or None, uh, min u per step)."""
    msh = mesh.create_rectangle(comm, [[-2, -2], [2, 2]], [N, N])
    V = fem.functionspace(msh, ("Lagrange", k))
    x = ufl.SpatialCoordinate(msh)
    t = fem.Constant(msh, 0.0)
    u_exact = 1 / (1 + 4 * AG * t) * ufl.exp(-AG * (x[0] ** 2 + x[1] ** 2) / (1 + 4 * AG * t))
    g = fem.Function(V); g_expr = fem.Expression(u_exact, V.element.interpolation_points)
    u_n = fem.Function(V)
    if init == "gauss":
        g.interpolate(g_expr); u_n.x.array[:] = g.x.array
    else:
        u_n.interpolate(lambda X: ((np.abs(X[0]) < 0.5) & (np.abs(X[1]) < 0.5)).astype(float))
    bc = fem.dirichletbc(g, boundary_dofs(msh, V))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    dtc, th = fem.Constant(msh, 1.0), fem.Constant(msh, 1.0)
    a = fem.form(u * v * ufl.dx + dtc * th * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx)
    L = fem.form(u_n * v * ufl.dx - dtc * (1 - th) * ufl.inner(ufl.grad(u_n), ufl.grad(v)) * ufl.dx)
    b, uh = create_vector(V), fem.Function(V)
    ksp = PETSc.KSP().create(comm); ksp.setType("preonly"); ksp.getPC().setType("lu")
    A, key, tt, mins = None, None, 0.0, []
    for dt, theta in zip(dts, thetas):
        if (dt, theta) != key:                          # assemble + factorise only when dt or theta changes
            dtc.value, th.value = dt, theta
            if A is not None:
                A.destroy()
            A = assemble_matrix(a, bcs=[bc]); A.assemble(); ksp.setOperators(A); key = (dt, theta)
        tt += dt
        t.value = tt
        if init == "gauss":
            g.interpolate(g_expr)                       # boundary data at the new time
        with b.localForm() as bl:
            bl.set(0.0)
        assemble_vector(b, L); apply_lifting(b, [a], [[bc]])
        b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE); set_bc(b, [bc])
        ksp.solve(b, uh.x.petsc_vec); uh.x.scatter_forward()
        u_n.x.array[:] = uh.x.array                     # copy, do not rebind
        mins.append(comm.allreduce(uh.x.array.min(), op=MPI.MIN))
    e = None
    if init == "gauss":
        e = np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form((uh - u_exact) ** 2 * ufl.dx)), op=MPI.SUM))
    ksp.destroy(); A.destroy(); b.destroy()
    return e, uh, mins

def schedule(nt, kind):
    """Time steps and thetas for BE, CN or CN with a Rannacher start (4 backward-Euler half steps)."""
    dt = T / nt
    if kind == "BE":
        return [dt] * nt, [1.0] * nt
    if kind == "CN":
        return [dt] * nt, [0.5] * nt
    return [dt / 2] * 4 + [dt] * (nt - 2), [1.0] * 4 + [0.5] * (nt - 2)

def profile(uh):
    """(x, u_h) on the line y = 0, gathered on rank 0 and sorted."""
    X = uh.function_space.tabulate_dof_coordinates()
    nown = uh.function_space.dofmap.index_map.size_local
    sel = np.abs(X[:nown, 1]) < 1e-10
    data = comm.gather(np.c_[X[:nown][sel, 0], uh.x.array[:nown][sel]], root=0)
    return np.vstack(data)[np.argsort(np.vstack(data)[:, 0])] if comm.rank == 0 else None

e, uh, mins = heat(64, 2, *schedule(20, "BE"))
umax = comm.allreduce(uh.x.array.max(), op=MPI.MAX)
say(f"BE, P2, N = 64, 20 steps: L2 error at T = {e:.3e};  max u_h = {umax:.4f} (exact 1/(1+4aT) = {1 / (1 + 4 * AG * T):.4f})")
p = profile(uh)
if comm.rank == 0:
    np.save("/tmp/fnx/profile_be20.npy", p)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p = np.load("/tmp/fnx/profile_be20.npy")
xs = np.linspace(-2, 2, 400); a, T = 5.0, 0.1
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(xs, np.exp(-a * xs ** 2), "k:", label="$u_0$")
ax.plot(xs, np.exp(-a * xs ** 2 / (1 + 4 * a * T)) / (1 + 4 * a * T), "k-", lw=2, label="exact at $T$")
ax.plot(p[:, 0], p[:, 1], "o", ms=3, label="BE, 20 steps")
ax.set_xlabel("$x$ (on $y=0$)"); ax.set_ylabel("$u$"); ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); plt.show()

## 2 · Temporal convergence  *(lecture §A6, §B4)*
Backward Euler on P2, $N=64$, $\Delta t=T/5,\dots,T/80$. The errors go to `/tmp/fnx/rates.npz`.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

AG, T = 5.0, 0.1

def heat(N, k, dts, thetas, init="gauss"):
    """theta-method, one step per entry of dts/thetas. init='gauss': exact Gaussian with its own Dirichlet data;
    'box': indicator of (-1/2,1/2)^2 with u = 0. Returns (L2 error at the end or None, uh, min u per step)."""
    msh = mesh.create_rectangle(comm, [[-2, -2], [2, 2]], [N, N])
    V = fem.functionspace(msh, ("Lagrange", k))
    x = ufl.SpatialCoordinate(msh)
    t = fem.Constant(msh, 0.0)
    u_exact = 1 / (1 + 4 * AG * t) * ufl.exp(-AG * (x[0] ** 2 + x[1] ** 2) / (1 + 4 * AG * t))
    g = fem.Function(V); g_expr = fem.Expression(u_exact, V.element.interpolation_points)
    u_n = fem.Function(V)
    if init == "gauss":
        g.interpolate(g_expr); u_n.x.array[:] = g.x.array
    else:
        u_n.interpolate(lambda X: ((np.abs(X[0]) < 0.5) & (np.abs(X[1]) < 0.5)).astype(float))
    bc = fem.dirichletbc(g, boundary_dofs(msh, V))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    dtc, th = fem.Constant(msh, 1.0), fem.Constant(msh, 1.0)
    a = fem.form(u * v * ufl.dx + dtc * th * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx)
    L = fem.form(u_n * v * ufl.dx - dtc * (1 - th) * ufl.inner(ufl.grad(u_n), ufl.grad(v)) * ufl.dx)
    b, uh = create_vector(V), fem.Function(V)
    ksp = PETSc.KSP().create(comm); ksp.setType("preonly"); ksp.getPC().setType("lu")
    A, key, tt, mins = None, None, 0.0, []
    for dt, theta in zip(dts, thetas):
        if (dt, theta) != key:                          # assemble + factorise only when dt or theta changes
            dtc.value, th.value = dt, theta
            if A is not None:
                A.destroy()
            A = assemble_matrix(a, bcs=[bc]); A.assemble(); ksp.setOperators(A); key = (dt, theta)
        tt += dt
        t.value = tt
        if init == "gauss":
            g.interpolate(g_expr)                       # boundary data at the new time
        with b.localForm() as bl:
            bl.set(0.0)
        assemble_vector(b, L); apply_lifting(b, [a], [[bc]])
        b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE); set_bc(b, [bc])
        ksp.solve(b, uh.x.petsc_vec); uh.x.scatter_forward()
        u_n.x.array[:] = uh.x.array                     # copy, do not rebind
        mins.append(comm.allreduce(uh.x.array.min(), op=MPI.MIN))
    e = None
    if init == "gauss":
        e = np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form((uh - u_exact) ** 2 * ufl.dx)), op=MPI.SUM))
    ksp.destroy(); A.destroy(); b.destroy()
    return e, uh, mins

def schedule(nt, kind):
    """Time steps and thetas for BE, CN or CN with a Rannacher start (4 backward-Euler half steps)."""
    dt = T / nt
    if kind == "BE":
        return [dt] * nt, [1.0] * nt
    if kind == "CN":
        return [dt] * nt, [0.5] * nt
    return [dt / 2] * 4 + [dt] * (nt - 2), [1.0] * 4 + [0.5] * (nt - 2)

def profile(uh):
    """(x, u_h) on the line y = 0, gathered on rank 0 and sorted."""
    X = uh.function_space.tabulate_dof_coordinates()
    nown = uh.function_space.dofmap.index_map.size_local
    sel = np.abs(X[:nown, 1]) < 1e-10
    data = comm.gather(np.c_[X[:nown][sel, 0], uh.x.array[:nown][sel]], root=0)
    return np.vstack(data)[np.argsort(np.vstack(data)[:, 0])] if comm.rank == 0 else None

NTS = [5, 10, 20, 40, 80]
E_be = [heat(64, 2, *schedule(nt, "BE"))[0] for nt in NTS]
say(f"BE, P2, N = 64: {[f'{e:.2e}' for e in E_be]}  rates {rate(E_be)}")
if comm.rank == 0:
    np.savez("/tmp/fnx/rates.npz", nts=NTS, be=E_be)

> **✏️ Your turn (10 min).** Repeat with Crank–Nicolson (`"CN"`) on $N=64$ **and** $N=128$, add the errors to `/tmp/fnx/rates_cn.npz` and plot all
> three curves (log–log) in the next cell. Predict the CN rate first. Where and why does the $N=64$ curve flatten?

In [ ]:
%%fenicsx -np 2
# TODO: CN on N = 64 and N = 128, save to /tmp/fnx/rates_cn.npz

In [ ]:
# TODO: log-log plot of the three error curves against dt

## 3 · Explicit Euler and its stability limit  *(lecture §A4–A5, §C1)*
Dense $M$ and $K$ on the interior DOFs (P1, serial — small meshes only), the generalised eigenvalues, $\Delta t_{\rm crit}=2/\lambda_{\max}$, and
explicit Euler at $0.95$ and $1.05\,\Delta t_{\rm crit}$ up to $t=1$ for the Gaussian and for box data.

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

import scipy.linalg as sla

def dense_MK(N, k=1, box=((-2, -2), (2, 2))):
    """Dense mass and stiffness matrices on the interior DOFs (serial), and the interior DOF coordinates."""
    msh = mesh.create_rectangle(MPI.COMM_SELF, [list(box[0]), list(box[1])], [N, N])
    V = fem.functionspace(msh, ("Lagrange", k))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    M = fem.assemble_matrix(fem.form(u * v * ufl.dx)).to_dense()
    K = fem.assemble_matrix(fem.form(ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx)).to_dense()
    I = np.setdiff1d(np.arange(M.shape[0]), boundary_dofs(msh, V))
    return M[np.ix_(I, I)], K[np.ix_(I, I)], V.tabulate_dof_coordinates()[I]

def explicit_euler(M, K, u0, dt, t_end):
    """M u^{n+1} = (M - dt K) u^n. Returns times and max|u| per step."""
    G = np.linalg.solve(M, M - dt * K)
    u, ts, ms = u0.copy(), [0.0], [np.abs(u0).max()]
    for n in range(1, int(np.ceil(t_end / dt)) + 1):
        u = G @ u; ts.append(n * dt); ms.append(np.abs(u).max())
    return np.array(ts), np.array(ms)

for N in (8, 16, 32):
    M, K, X = dense_MK(N)
    lam = sla.eigh(K, M, eigvals_only=True); h = 4 / N
    print(f"N = {N:2d}: lam_max = {lam.max():7.1f}, lam_max h^2 = {lam.max() * h * h:.2f}, dt_crit = {2 / lam.max():.3e}")
dtc = 2 / lam.max()
out = {}
for init in ("gauss", "box"):
    u0 = np.exp(-5 * (X[:, 0] ** 2 + X[:, 1] ** 2)) if init == "gauss" else ((np.abs(X[:, 0]) < 0.5) & (np.abs(X[:, 1]) < 0.5)).astype(float)
    for fac in (0.95, 1.05):
        ts, ms = explicit_euler(M, K, u0, fac * dtc, 1.0)
        out[f"{init}_{fac}_t"], out[f"{init}_{fac}_m"] = ts, ms
        print(f"{init:5s}, dt = {fac} dt_crit: max|u| at t = 0.1, 0.5, 1: " + ", ".join(f"{ms[int(np.ceil(tq / (fac * dtc)))]:.2e}" for tq in (0.1, 0.5, 1.0)))
np.savez("/tmp/fnx/explicit.npz", **out)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = np.load("/tmp/fnx/explicit.npz")
fig, ax = plt.subplots(figsize=(6.2, 4))
for key, st in (("gauss_0.95", "-"), ("gauss_1.05", "--"), ("box_0.95", "-."), ("box_1.05", ":")):
    ax.semilogy(d[key + "_t"], d[key + "_m"], st, lw=2, label=key.replace("_", ", dt = ") + " dt_crit")
ax.set_ylim(1e-2, 1e20); ax.set_xlabel("$t$"); ax.set_ylabel(r"$\max|u_h|$"); ax.grid(alpha=0.3, which="both"); ax.legend(fontsize=9)
fig.tight_layout(); plt.show()

> **✏️ Your turn (10 min).** **Mass lumping** replaces $M$ by the diagonal matrix of its row sums. Compute $\lambda_{\max}$ and $\Delta t_{\rm crit}$ with the
> lumped mass for $N=8,16,32$ and the ratio to the consistent-mass $\Delta t_{\rm crit}$. Why does lumping help an explicit method — and what does it cost?

In [ ]:
%%fenicsx -np 1
# TODO: lumped-mass lam_max and dt_crit

## 4 · Non-smooth data: Crank–Nicolson oscillates  *(lecture §C2–C3)*
Box initial data, $u=0$ on $\partial\Omega$, P1, $N=32$, four steps ($\Delta t=T/4$): backward Euler, Crank–Nicolson, and CN with a Rannacher start
(four backward-Euler half steps). Reference: Rannacher-started CN with 4096 steps.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

AG, T = 5.0, 0.1

def heat(N, k, dts, thetas, init="gauss"):
    """theta-method, one step per entry of dts/thetas. init='gauss': exact Gaussian with its own Dirichlet data;
    'box': indicator of (-1/2,1/2)^2 with u = 0. Returns (L2 error at the end or None, uh, min u per step)."""
    msh = mesh.create_rectangle(comm, [[-2, -2], [2, 2]], [N, N])
    V = fem.functionspace(msh, ("Lagrange", k))
    x = ufl.SpatialCoordinate(msh)
    t = fem.Constant(msh, 0.0)
    u_exact = 1 / (1 + 4 * AG * t) * ufl.exp(-AG * (x[0] ** 2 + x[1] ** 2) / (1 + 4 * AG * t))
    g = fem.Function(V); g_expr = fem.Expression(u_exact, V.element.interpolation_points)
    u_n = fem.Function(V)
    if init == "gauss":
        g.interpolate(g_expr); u_n.x.array[:] = g.x.array
    else:
        u_n.interpolate(lambda X: ((np.abs(X[0]) < 0.5) & (np.abs(X[1]) < 0.5)).astype(float))
    bc = fem.dirichletbc(g, boundary_dofs(msh, V))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    dtc, th = fem.Constant(msh, 1.0), fem.Constant(msh, 1.0)
    a = fem.form(u * v * ufl.dx + dtc * th * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx)
    L = fem.form(u_n * v * ufl.dx - dtc * (1 - th) * ufl.inner(ufl.grad(u_n), ufl.grad(v)) * ufl.dx)
    b, uh = create_vector(V), fem.Function(V)
    ksp = PETSc.KSP().create(comm); ksp.setType("preonly"); ksp.getPC().setType("lu")
    A, key, tt, mins = None, None, 0.0, []
    for dt, theta in zip(dts, thetas):
        if (dt, theta) != key:                          # assemble + factorise only when dt or theta changes
            dtc.value, th.value = dt, theta
            if A is not None:
                A.destroy()
            A = assemble_matrix(a, bcs=[bc]); A.assemble(); ksp.setOperators(A); key = (dt, theta)
        tt += dt
        t.value = tt
        if init == "gauss":
            g.interpolate(g_expr)                       # boundary data at the new time
        with b.localForm() as bl:
            bl.set(0.0)
        assemble_vector(b, L); apply_lifting(b, [a], [[bc]])
        b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE); set_bc(b, [bc])
        ksp.solve(b, uh.x.petsc_vec); uh.x.scatter_forward()
        u_n.x.array[:] = uh.x.array                     # copy, do not rebind
        mins.append(comm.allreduce(uh.x.array.min(), op=MPI.MIN))
    e = None
    if init == "gauss":
        e = np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form((uh - u_exact) ** 2 * ufl.dx)), op=MPI.SUM))
    ksp.destroy(); A.destroy(); b.destroy()
    return e, uh, mins

def schedule(nt, kind):
    """Time steps and thetas for BE, CN or CN with a Rannacher start (4 backward-Euler half steps)."""
    dt = T / nt
    if kind == "BE":
        return [dt] * nt, [1.0] * nt
    if kind == "CN":
        return [dt] * nt, [0.5] * nt
    return [dt / 2] * 4 + [dt] * (nt - 2), [1.0] * 4 + [0.5] * (nt - 2)

def profile(uh):
    """(x, u_h) on the line y = 0, gathered on rank 0 and sorted."""
    X = uh.function_space.tabulate_dof_coordinates()
    nown = uh.function_space.dofmap.index_map.size_local
    sel = np.abs(X[:nown, 1]) < 1e-10
    data = comm.gather(np.c_[X[:nown][sel, 0], uh.x.array[:nown][sel]], root=0)
    return np.vstack(data)[np.argsort(np.vstack(data)[:, 0])] if comm.rank == 0 else None

_, uref, _ = heat(32, 1, *schedule(4096, "R"), init="box")
P = {"ref": profile(uref)}
for kind in ("BE", "CN", "R"):
    _, uh, mins = heat(32, 1, *schedule(4, kind), init="box")
    d = fem.Function(uh.function_space); d.x.array[:] = uh.x.array - uref.x.array          # same mesh and partition
    err = np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form(d ** 2 * ufl.dx)), op=MPI.SUM))
    say(f"{kind:2s}, 4 steps: error vs reference {err:.2e}, min u over the steps {min(mins):.2e}")
    P[kind] = profile(uh)
if comm.rank == 0:
    np.savez("/tmp/fnx/nonsmooth.npz", **P)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

P = np.load("/tmp/fnx/nonsmooth.npz")
fig, ax = plt.subplots(figsize=(6.8, 4))
ax.plot(P["ref"][:, 0], P["ref"][:, 1], "k-", lw=2.5, label="reference")
for k, lab in (("BE", "BE"), ("CN", "CN"), ("R", "CN + Rannacher")):
    ax.plot(P[k][:, 0], P[k][:, 1], "o-", ms=3, lw=1.2, label=lab)
ax.set_xlabel("$x$ (on $y=0$)"); ax.set_ylabel("$u_h(x,0,T)$"); ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); plt.show()

> **✏️ Your turn (10 min).** For $\Delta t=T/4,T/8,\dots,T/64$ compute the error against the reference for BE, CN and CN + Rannacher, and the rates.
> When does CN become second order? Explain with $R(z)$ (lecture §A5).

In [ ]:
%%fenicsx -np 2
# TODO: errors vs reference for nt = 4 ... 64

## 5 · Cahn–Hilliard: θ-method + Newton  *(lecture §D1)*
$c_t=\nabla\cdot\nabla\mu$, $\mu=f'(c)-\lambda\Delta c$, $f=100c^2(1-c)^2$ on the unit square ($64\times64$, P1×P1, $\lambda=10^{-2}$, $\Delta t=5\times10^{-6}$,
θ = ½, 80 steps). The initial state $c_0=0.63\pm0.01$ is a deterministic pseudo-random field — the same for any number of processes.
Snapshots and the free energy go to `/tmp/fnx/ch.npz` (this cell takes about a minute).

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

from basix.ufl import element, mixed_element
from dolfinx import default_real_type
from dolfinx.fem.petsc import NonlinearProblem

msh = mesh.create_unit_square(comm, 64, 64)
P1 = element("Lagrange", msh.basix_cell(), 1, dtype=default_real_type)
ME = fem.functionspace(msh, mixed_element([P1, P1]))
q, v = ufl.TestFunctions(ME)
u, u0 = fem.Function(ME), fem.Function(ME)
c, mu = ufl.split(u); c0, mu0 = ufl.split(u0)
lmbda, dt, theta = 1.0e-2, 5.0e-6, 0.5

def noise(X):
    s = np.sin(12.9898 * X[0] + 78.233 * X[1]) * 43758.5453
    return 0.63 + 0.02 * (0.5 - (s - np.floor(s)))

u.sub(0).interpolate(noise); u.x.scatter_forward()
cv = ufl.variable(c)
fdens = 100 * cv ** 2 * (1 - cv) ** 2
mu_mid = (1 - theta) * mu0 + theta * mu
F = (c * q - c0 * q + dt * ufl.inner(ufl.grad(mu_mid), ufl.grad(q))) * ufl.dx \
    + (mu * v - ufl.diff(fdens, cv) * v - lmbda * ufl.inner(ufl.grad(c), ufl.grad(v))) * ufl.dx
problem = NonlinearProblem(F, u, petsc_options_prefix="ch_", petsc_options={"snes_type": "newtonls", "snes_linesearch_type": "none",
                           "snes_rtol": 1e-8, "snes_atol": 1e-10, "ksp_type": "preonly", "pc_type": "lu"})
energy = fem.form((100 * c ** 2 * (1 - c) ** 2 + lmbda / 2 * ufl.inner(ufl.grad(c), ufl.grad(c))) * ufl.dx)
S = lambda form: comm.allreduce(fem.assemble_scalar(form), op=MPI.SUM)
V0, m0 = ME.sub(0).collapse(); m0 = np.asarray(m0).reshape(-1)       # the map comes back as a list of arrays
X0, nown = V0.tabulate_dof_coordinates(), V0.dofmap.index_map.size_local

def snapshot():
    data = comm.gather(np.c_[X0[:nown, :2], u.x.array[m0[:nown]]], root=0)
    return np.vstack(data) if comm.rank == 0 else None

snaps, E, its = {"s0": snapshot()}, [S(energy)], []
for n in range(1, 81):
    u0.x.array[:] = u.x.array                       # previous step (copy)
    problem.solve()
    assert problem.solver.getConvergedReason() > 0
    its.append(problem.solver.getIterationNumber()); E.append(S(energy))
    if n in (10, 80):
        snaps[f"s{n}"] = snapshot()
say(f"free energy {E[0]:.4f} -> {E[10]:.4f} (10 steps) -> {E[-1]:.4f} (80 steps); Newton iterations per step {min(its)}-{max(its)}")
if comm.rank == 0:
    np.savez("/tmp/fnx/ch.npz", E=np.array(E), **snaps)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = np.load("/tmp/fnx/ch.npz")
fig, axs = plt.subplots(1, 4, figsize=(14, 3.3))
for ax, k in zip(axs[:3], ("s0", "s10", "s80")):
    s = d[k]
    ax.tricontourf(s[:, 0], s[:, 1], s[:, 2], levels=20, cmap="viridis"); ax.set_aspect("equal"); ax.set_title(k.replace("s", "step "))
    ax.set_xticks([]); ax.set_yticks([])
axs[3].plot(d["E"]); axs[3].set_xlabel("step"); axs[3].set_ylabel("free energy"); axs[3].grid(alpha=0.3)
fig.tight_layout(); plt.show()

> **✏️ Your turn (5 min).** There is no exact solution. Check two properties every step: $\int_\Omega c\,dx$ (print it at steps 0, 40, 80) and
> $E^{n+1}\le E^n$ (`np.all(np.diff(E) < 0)` on rank 0). Why is $\int c$ conserved **exactly** by the discrete scheme? (Hint: choose $q=1$.)

In [ ]:
%%fenicsx -np 2
# TODO: mass and energy checks for Cahn-Hilliard

## 6 · LLM-assisted coding: live demo  *(lecture §D2)*
Prompt (paste the API reference sheet first):

```text
Context: [API reference sheet]
Task: DOLFINx 0.11, one %%fenicsx -np 2 cell. Solve u_t = Lap u on (-2,2)^2 with P2 on a 64x64 mesh, Crank-Nicolson, T = 0.1, 20 steps.
Initial and Dirichlet data from the exact solution u = exp(-5(x^2+y^2)/(1+20t))/(1+20t) (time-dependent boundary values).
Assemble the matrix once, reuse the LU factorisation, print the L2 error at T on rank 0.
```
Paste the result below **unchanged**, run it, then go through the checklist.

In [ ]:
%%fenicsx -np 2
# TODO: paste the %%fenicsx part your LLM produced (unchanged) and run it

**Verification checklist** — checks with a known answer

- [ ] $t$ (and $\Delta t$) are `fem.Constant`s; the boundary function is re-interpolated **every** step at $t_{n+1}$
- [ ] $u_n$ is updated by copying values (`u_n.x.array[:] = uh.x.array`), not by `u_n = uh`
- [ ] The loop runs over an integer step counter; the final time is exactly $T$ (print it)
- [ ] The matrix is assembled and factorised once (no `LinearProblem(...)`/`fem.form(...)` inside the loop)
- [ ] The error is computed against the exact solution **at the same time**; for 20 CN steps it is $6.9\times10^{-5}$ (lecture §B4)